<a href="https://colab.research.google.com/github/Harshithpalan/Python-projects/blob/main/detect_toxic_comments_in_online_discussions_.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Toxic Comment Detection Model
This notebook develops a machine learning model to identify toxicity in online comments (e.g., toxic, severe toxic, obscene, threat, insult, identity hate).

In [1]:
import pandas as pd
import numpy as np
import tensorflow as tf
from tensorflow.keras.layers import TextVectorization, Embedding, Bidirectional, LSTM, Dense
from tensorflow.keras.models import Sequential
import matplotlib.pyplot as plt

# Configure GPU memory growth if available
gpus = tf.config.list_physical_devices('GPU')
if gpus:
    try:
        for gpu in gpus:
            tf.config.experimental.set_memory_growth(gpu, True)
    except RuntimeError as e:
        print(e)

### 1. Data Preparation
In a real scenario, you would load a CSV here. For this demonstration, I will create a placeholder structure to show how to vectorize text.

In [2]:
# Example: Defining the maximum number of words and sequence length
MAX_FEATURES = 200000 # number of words in the vocabulary
MAX_LEN = 128         # max length of a comment in words

vectorizer = TextVectorization(max_tokens=MAX_FEATURES,
                               output_sequence_length=MAX_LEN,
                               output_mode='int')

# Sample data for demonstration
sample_comments = ["I love this!", "You are so annoying and I hate you.", "This is a great tutorial.", "Go away, nobody likes you!"]
vectorizer.adapt(sample_comments)

def preprocess_text(text):
    return vectorizer(text)

### 2. Model Architecture
We'll use a Bidirectional LSTM model, which is effective for understanding context in text sequence data.

In [3]:
model = Sequential([
    Embedding(MAX_FEATURES + 1, 32),
    Bidirectional(LSTM(32, activation='tanh')),
    Dense(128, activation='relu'),
    Dense(64, activation='relu'),
    Dense(6, activation='sigmoid') # 6 outputs for different toxicity classes
])

model.compile(loss='BinaryCrossentropy', optimizer='Adam', metrics=['accuracy'])
model.summary()

Model: "sequential"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ embedding (Embedding)           │ ?                      │   0 (unbuilt) │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ bidirectional (Bidirectional)   │ ?                      │   0 (unbuilt) │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense (Dense)                   │ ?                      │   0 (unbuilt) │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_1 (Dense)                 │ ?                      │   0 (unbuilt) │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_2 (Dense)                 │ ?                      │   0 (unbuilt) │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 0 (0.00 B)

 Trainable params: 0 (0.00 B)

 Non-trainable params: 0 (0.00 B)

### 3. Prediction Function
Once trained, you can use this function to check new comments.

In [4]:
def score_comment(comment):
    vectorized_comment = vectorizer([comment])
    results = model.predict(vectorized_comment)

    labels = ['toxic', 'severe_toxic', 'obscene', 'threat', 'insult', 'identity_hate']
    return {label: results[0][idx] for idx, label in enumerate(labels)}

# Test with a dummy prediction
test_res = score_comment("You are a wonderful person!")
print(test_res)

1/1 ━━━━━━━━━━━━━━━━━━━━ 1s 1s/step
{'toxic': np.float32(0.49973482), 'severe_toxic': np.float32(0.49854174), 'obscene': np.float32(0.49983177), 'threat': np.float32(0.50070107), 'insult': np.float32(0.5011918), 'identity_hate': np.float32(0.5007888)}
